# Custom Knowledge Assistant with LangChain + OpenAI + MLflow

This notebook **reimplements** the same capability as Databricks Knowledge Assistant using custom libraries:
- **Data**: PDF flyers from `data/flyers` (table + catalog style)
- **Segmentation**: Chunk documents for retrieval
- **Vector space**: Databricks embedding endpoint + Chroma vector store
- **LLM**: Databricks chat endpoint (OpenAI-compatible API) for Q&A over the docs
- **Observability**: MLflow experiment tracking + tracing

**Purpose**: Show how much custom code is required compared to using Databricks Knowledge Assistant (which handles ingestion, chunking, embedding, indexing, and serving in a few steps).

## 1. Install dependencies

Run once (or use the project venv with these packages installed).

In [ ]:
# Uncomment to install in current env:
# %pip install langchain langchain-openai langchain-community pypdf chromadb 'mlflow[genai]' openai

## 2. Imports and configuration

In [ ]:
import os
import json
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import mlflow

# Paths: assume notebook is in flipp_agent/notebooks/, data is in flipp_agent/data/flyers
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
FLYER_DIR = PROJECT_ROOT / "data" / "flyers"
TABLE_FLYERS = FLYER_DIR / "table_style"
CATALOG_FLYERS = FLYER_DIR / "catalog_style"

# Databricks endpoints (OpenAI-compatible API)
# Set DATABRICKS_TOKEN (e.g. from Profile → Generate Access Token) and optionally DATABRICKS_BASE_URL
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
DATABRICKS_BASE_URL = os.environ.get("DATABRICKS_BASE_URL") or os.environ.get("DATABRICKS_AI_GATEWAY_URL")
if not DATABRICKS_BASE_URL:
    # Example: https://<workspace-id>.ai-gateway.cloud.databricks.com/mlflow/v1
    DATABRICKS_BASE_URL = "https://7474654018043568.ai-gateway.cloud.databricks.com/mlflow/v1"
assert DATABRICKS_TOKEN, "Set DATABRICKS_TOKEN in environment"

# Endpoint names (your Databricks model serving endpoint names for embeddings and chat)
EMBEDDING_MODEL = os.environ.get("DATABRICKS_EMBEDDING_MODEL", "text-embedding-3-small")
LLM_MODEL = os.environ.get("DATABRICKS_LLM_MODEL", "databricks-gpt-5-mini")

# Chunking and retrieval (we will log these to MLflow)
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K_RETRIEVAL = 6

## 3. Load documents from data/flyers

Load all PDFs from table_style and catalog_style. Optionally attach JSON metadata (retailer, validity) to each document.

In [ ]:
def load_flyer_docs(pdf_dirs):
    """Load PDFs from given directories; optionally merge JSON metadata per flyer."""
    docs = []
    for directory in pdf_dirs:
        if not directory.exists():
            continue
        for pdf_path in sorted(directory.glob("*.pdf")):
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
            # Attach source and optional JSON metadata
            json_path = pdf_path.with_suffix(".json")
            meta_extra = {}
            if json_path.exists():
                with open(json_path) as f:
                    meta_extra = json.load(f)
            for i, page in enumerate(pages):
                page.metadata["source"] = str(pdf_path)
                page.metadata["page"] = i + 1
                for k, v in meta_extra.items():
                    if k not in page.metadata and v is not None:
                        page.metadata[k] = str(v)
                docs.append(page)
    return docs

documents = load_flyer_docs([TABLE_FLYERS, CATALOG_FLYERS])
print(f"Loaded {len(documents)} pages from flyer PDFs.")

## 4. Segment (chunk) documents

Split text into overlapping chunks for retrieval. Size and overlap affect recall and context.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)
print(f"Segmented into {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}).")

## 5. Create embeddings and vector store

Embed each chunk with OpenAI and add to Chroma. This is the "vector space" and index for retrieval.

In [ ]:
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=DATABRICKS_TOKEN,
    openai_api_base=DATABRICKS_BASE_URL,
)

# Persist to disk so we can reuse without re-embedding every run (optional)
CHROMA_DIR = PROJECT_ROOT / "chroma_flyers"
CHROMA_DIR.mkdir(exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name="flyer_deals",
)
vectorstore.persist()

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K_RETRIEVAL})
print(f"Vector store ready: {len(chunks)} chunks, top_k={TOP_K_RETRIEVAL}.")

## 6. Build RAG chain (retriever + prompt + LLM)

Chain: user question → retriever → format context → prompt + LLM → answer.

In [ ]:
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about grocery flyer deals using only the context below. "
     "Include prices and retailer/validity when available. If the context does not contain relevant deals, say so."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0.2,
    openai_api_key=DATABRICKS_TOKEN,
    openai_api_base=DATABRICKS_BASE_URL,
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built (retriever -> prompt -> OpenAI -> answer).")

## 7. MLflow: experiment, params, and tracing

Create an experiment, start a run, log hyperparameters, and enable LangChain autolog so each chain invocation is traced.

In [ ]:
mlflow.set_tracking_uri("mlruns" if (PROJECT_ROOT / "mlruns").exists() else None)  # local ./mlruns or default
mlflow.set_experiment("custom_ka_flyer_rag")

# Trace LangChain chain invocations automatically
mlflow.langchain.autolog(log_traces=True, log_models=False)

with mlflow.start_run(run_name="flyer_rag_run") as run:
    mlflow.log_params({
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "embedding_model": EMBEDDING_MODEL,
        "llm_model": LLM_MODEL,
        "top_k": TOP_K_RETRIEVAL,
        "n_chunks": len(chunks),
        "n_docs": len(documents),
    })
    run_id = run.info.run_id
    print(f"MLflow run started: {run_id}")

    # Query inside the run so the chain invocation is traced
    question = "What vegetables or produce are on sale?"
    answer = rag_chain.invoke(question)
    print("\nQuestion:", question)
    print("Answer:", answer)
    mlflow.log_metric("num_queries_demo", 1)

## 8. View traces and run more queries (optional)

The first query ran inside the MLflow run above (so it is traced). Run `mlflow ui` in the project root to view traces. Below: more example queries.

In [ ]:
# Optional: run more queries (each invoke will be traced if autolog is active)
print("To view traces: run `mlflow ui` in the project root and open the run.")
print("Example: rag_chain.invoke('Are there any halal meat deals?')")

## 9. Optional: run more queries and log as metrics

You can log each query/answer or evaluate and log metrics before ending the run.

In [ ]:
more_questions = [
    "Are there any halal meat deals?",
    "What are the best deals for a barbecue?",
]
for q in more_questions:
    out = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {out[:200]}..." if len(out) > 200 else f"A: {out}")
    print("-" * 60)

# Log number of queries as a simple metric
mlflow.log_metric("num_queries_demo", 1 + len(more_questions))

---

**Summary**: This single notebook implements document loading, segmentation, embedding, vector store, RAG chain, and MLflow tracking. Doing the same with **Databricks Knowledge Assistant** only requires pointing a volume path and creating the KA tile—no chunking, embedding, or vector store code. The custom LangChain approach is flexible but requires significantly more code and maintenance.